In [ ]:
# A1_R50_1536 R2. Attach official competition data, g-data v1, Q1 v4output v2, and original v1output.
# The official submission schema is verified without requiring a separate sample_submission.csv.
# Accelerator: GPU. With multiple cards, this Notebook uses one card.
# mode: audit = exact baseline only; full = train and export; inference = reload an existing best checkpoint.
# Optional: mount the original complete Fusion output to load the original C_HGB artifact.
# For combined comparison, set peer_result_root to the mounted sibling experiment output directory.
from pathlib import Path
import base64
import gzip
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys

CONFIG = {'input_root': '/kaggle/input', 'output_root': '/kaggle/working/solar_A_line', 'v1_source': '', 'fusion_source': '', 'mode': 'full', 'max_hours': 10.5, 'workers': 2, 'allow_imagenet_download': True, 'peer_result_root': ''}
ARM = 'A1'
FUSION = {'input_root': '/kaggle/input', 'output_root': '/kaggle/working/solar_g3_q1_fusion', 'seed': 15026, 'max_hours': 10.5, 'cv_folds': 3, 'install_missing': True, 'stability_deltas': (0.025, 0.05, 0.1), 'stability_roi_margin': 16, 'taus': (0.16, 0.2, 0.24, 0.28, 0.32, 0.36), 'orphan_floors': (0.5, 0.6, 0.7), 'expected_experiments': {'g': '64a521227303a75f03aa3e7123b28bfd34a0b7db695265f9771794dc105b40b0', 'q': 'c66397b570b02f89bbae9a3fa774a0f1536341e64fbbe8d045f0d9875bf68ba0'}, 'expected_reference_hashes': {'g': {'candidate_decisions.csv': 'e7c56e61d521abc23e1ba708b321cb24845f3dae817a48d497bdf2fa76d79082', 'submission_G3.csv': '1f97a7fecedd66a17bab42d5d83eafa418656d060137102c83ad866f324a7a87', 'gt_fates.csv': '8a09ae0e1fd722f06cfeb4450bb8b32b4f6bb119274ad7cc6f146f260dd6aa4a', 'exposure_roles.json': 'c94c7256811bc8140f9c1c7cc2cc226e65b51971fc203aa31747e281c933c60e', 'submission_G4.csv': '1f97a7fecedd66a17bab42d5d83eafa418656d060137102c83ad866f324a7a87', 'models/G3.joblib': '9a3f048b60b80278ee76e5928eaba68ad62304969b4187fb3331156cdca7d07f'}, 'q': {'gt_fates.csv': 'bbca34c17a8642d195a9ffa671130b199d55e18848edd39c997d4d40624fa4e1', 'exposure_roles.json': 'c94c7256811bc8140f9c1c7cc2cc226e65b51971fc203aa31747e281c933c60e', 'submission_Q1.csv': 'a4fe75491753e1855da9d907f5319200feafba565fd0c5226e68a66e45ac0b6b', 'candidate_decisions.csv': 'a139cd3ba79ec7482153ba75435fa54e7164ee4ab09b06939aa1be456485a6df', 'models/Q1.joblib': 'a087854ef7ede88ea7c18f46f42c0e06c4265ef4c112ccd9d7a9a84b35a3c8a7'}}, 'v1_config_sha256': '178a71893cda80b464eefe53a201d0e3dc426bc2865f09ec59c5586c86f60c16', 'v1_thresholds_sha256': '1cf97acb2ee69e653005e9382cebf05a7684e82d28885734b55d286bfb58696b'}
FUSION_CODE = '950d83d568f96468b66954dce8509268c65976a2ab33430ff2d388d57075ff27'
CODE_SHA = 'bb232e2bfb112f9b7d75a07070207d22ab31bc2f94f219c5757614622ac6374b'
MODULE_PAYLOAD = '...'
SPLIT_PAYLOAD = '...'

versions = {'numpy':'2.0.2', 'scipy':'1.16.3', 'scikit-image':'0.25.2', 'scikit-learn':'1.6.1'}
install = []
for package, version in versions.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install.append(f'{package}=={version}')
for module, spec in {'pycocotools':'pycocotools==2.0.11', 'psutil':'psutil',
                     'joblib':'joblib','threadpoolctl':'threadpoolctl'}.items():
    if importlib.util.find_spec(module) is None:
        install.append(spec)
if install:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *install], check=True)
opencv = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'], capture_output=True, text=True)
if opencv.returncode or opencv.stdout.strip() != '4.13.0':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless==4.13.*'], check=True)
subprocess.run([sys.executable, '-c', 'import cv2; assert cv2.__version__ == "4.13.0", cv2.__version__'], check=True)

payload = gzip.decompress(base64.b64decode(MODULE_PAYLOAD))
if hashlib.sha256(payload).hexdigest() != CODE_SHA:
    raise ValueError('Packaged module fingerprint mismatch')
runtime = Path('/kaggle/temp/solar_a_line_' + CODE_SHA[:16])
for name, source in json.loads(payload).items():
    path = runtime / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8', newline='\n')
split = runtime / 'validation_split.json'
split.write_bytes(gzip.decompress(base64.b64decode(SPLIT_PAYLOAD)))
env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONHASHSEED='0',
           OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
entry = 'import json, sys; from solar_a_line.run import run; run(sys.argv[1], json.loads(sys.argv[2]), json.loads(sys.argv[3]), *sys.argv[4:])'
subprocess.run([sys.executable, '-c', entry, ARM, json.dumps(CONFIG), json.dumps(FUSION),
                FUSION_CODE, CODE_SHA, str(split)], cwd=runtime, env=env, check=True)
